In [0]:
# Databricks notebook source

# ============================================================
# run_dq.py — Data Quality checks (after Silver, before Gold)
#
# Upload to: /Shared/Capstone/run_dq.py
#
# Add to batch_pipeline DAG between run_silver and run_gold:
#   run_bronze → run_silver → run_dq → run_gold
#
# What this does:
#   1. Reads each Silver transactional table
#   2. Runs typed DQ checks (null / range / duplicate / referential)
#   3. Appends all results → silver._dq_log (Delta)
#   4. Quarantines bad rows → silver._dq_quarantine (Delta)
#   5. Overwrites Silver table with clean rows only
#   6. Prints pass/fail summary
#   7. Raises exception ONLY if an entire table is 100% bad
#
# Column names match table_config.py exactly — no guesses.
#
# Reference tables (customers, sellers, products, geolocation,
# category_translation) are skipped — they are authoritative
# CSV loads and not subject to row-level DQ quarantine.
# ============================================================

import sys
import importlib

sys.path.append("/Workspace/Shared/Capstone_final/Databricks")

import table_config
importlib.reload(table_config)
import utils
importlib.reload(utils)


from table_config import S3_DELTA_SILVER, S3_BUCKET, S3_PROTOCOL
from pyspark.sql.functions import col, lit, current_timestamp

# ── Widget (matches run_silver.py pattern) ────────────────────────────────────
try:
    batch_number = dbutils.widgets.get("batch_number")
except Exception:
    dbutils.widgets.text("batch_number", "1")
    batch_number = "1"

batch_id = f"batch_{batch_number}" if batch_number != "live" else "live_stream"

print("=" * 60)
print(f"DQ ENGINE — Silver validation before Gold")
print(f"Batch     : {batch_number}  (batch_id: {batch_id})")
print("=" * 60)

# ── Output Delta paths (alongside Silver tables) ──────────────────────────────
DQ_LOG_PATH        = f"{S3_PROTOCOL}://{S3_BUCKET}/delta/silver/_dq_log"
DQ_QUARANTINE_PATH = f"{S3_PROTOCOL}://{S3_BUCKET}/delta/silver/_dq_quarantine"

all_results   = []
stop_pipeline = False


# ── DQ check runner ───────────────────────────────────────────────────────────
def run_check(test_name, table_name, sql):
    """
    Run one DQ check SQL against a registered Spark temp view.
    SQL must return exactly one row, one column named 'result'.
      result = 0  → PASS
      result > 0  → FAIL (number of bad rows)
      result = -1 → ERROR (exception during execution)
    """
    try:
        result_value = spark.sql(sql).collect()[0]["result"]
        status = "PASS" if int(result_value) == 0 else "FAIL"
        marker = "✓" if status == "PASS" else "✗"
        print(f"  [{status:4s}] {marker} {test_name}: {result_value} bad rows")
        return {
            "test_name"   : test_name,
            "table_name"  : table_name,
            "result_value": int(result_value),
            "status"      : status,
            "batch_id"    : batch_id,
        }
    except Exception as e:
        print(f"  [ERROR] ✗ {test_name}: {e}")
        return {
            "test_name"   : test_name,
            "table_name"  : table_name,
            "result_value": -1,
            "status"      : "ERROR",
            "batch_id"    : batch_id,
        }


# ── Quarantine + clean Silver overwrite ───────────────────────────────────────
def quarantine_and_clean(table_name, df_silver, bad_condition, table_results):
    """
    Only runs if at least one check FAILed for this table.
    Bad rows → silver._dq_quarantine (Delta, append)
    Good rows → silver.<table_name> (Delta, overwrite in-place)
    Returns True  → pipeline continues (good rows exist)
    Returns False → ALL rows are bad (pipeline stops after summary)
    """
    global stop_pipeline

    has_failures = any(r["status"] == "FAIL" for r in table_results)
    if not has_failures:
        print(f"  ✓ {table_name}: all checks passed — no quarantine needed")
        return True

    df_bad = (
        df_silver.filter(bad_condition)
        .withColumn("quarantine_reason",    lit("failed_dq_checks"))
        .withColumn("quarantine_table",     lit(table_name))
        .withColumn("quarantine_batch_id",  lit(batch_id))
        .withColumn("quarantine_timestamp", current_timestamp())
    )
    df_good = df_silver.filter(~bad_condition)

    total_count = df_silver.count()
    bad_count   = df_bad.count()
    good_count  = df_good.count()
    pct_bad     = round(bad_count / total_count * 100, 2) if total_count > 0 else 0

    print(f"\n  QUARANTINE — {table_name}")
    print(f"    Total : {total_count:,}")
    print(f"    Bad   : {bad_count:,}  ({pct_bad}%)")
    print(f"    Clean : {good_count:,}")

    # Append bad rows to quarantine Delta table
    (
        df_bad.write
        .format("delta")
        .mode("append")
        .option("path", DQ_QUARANTINE_PATH)
        .option("mergeSchema", "true")
        .saveAsTable("silver._dq_quarantine")
    )
    print(f"    Bad rows saved → silver._dq_quarantine")

    if good_count == 0:
        print(f"\n  ⛔ ALL rows in {table_name} are bad — pipeline will stop after summary.")
        stop_pipeline = True
        return False

    # Overwrite Silver in-place with clean rows only
    # gold_engine._read_silver() reads these paths directly
    s3_path = f"{S3_DELTA_SILVER}/{table_name}"
    spark.sql(f"DROP TABLE IF EXISTS silver.{table_name}")
    (
        df_good.write
        .format("delta")
        .mode("overwrite")
        .option("path", s3_path)
        .option("overwriteSchema", "true")
        .saveAsTable(f"silver.{table_name}")
    )
    print(f"    Clean Silver saved → silver.{table_name} ({good_count:,} rows)")
    print(f"    Gold will receive {good_count:,} clean rows.")
    return True


# ══════════════════════════════════════════════════════════════════════════════
# TABLE 1: silver.orders
#
# Source: orders_dataset.csv
# table_config cleaning_rules applied in Silver:
#   order_purchase_timestamp     → to_timestamp
#   order_approved_at            → to_timestamp
#   order_delivered_carrier_date → to_timestamp
#   order_delivered_customer_date→ to_timestamp
#   order_estimated_delivery_date→ to_timestamp
#
# All timestamp columns arrive in Silver as TIMESTAMP (or NULL if cast failed).
# Merge key: order_id
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 60)
print("TABLE: silver.orders")
print("─" * 60)

try:
    df_orders = spark.read.format("delta").load(f"{S3_DELTA_SILVER}/orders")
    df_orders.createOrReplaceTempView("sv_orders")
    orders_ok = True
except Exception as e:
    print(f"  [SKIP] Cannot read silver.orders: {e}")
    orders_ok = False

if orders_ok:
    t = []

    t.append(run_check(
        "orders_no_null_order_id", "orders",
        "SELECT COUNT(*) AS result FROM sv_orders WHERE order_id IS NULL"
    ))
    t.append(run_check(
        "orders_no_null_customer_id", "orders",
        "SELECT COUNT(*) AS result FROM sv_orders WHERE customer_id IS NULL"
    ))
    t.append(run_check(
        "orders_no_null_purchase_timestamp", "orders",
        # NULL here = Silver to_timestamp cast failed on original string
        "SELECT COUNT(*) AS result FROM sv_orders WHERE order_purchase_timestamp IS NULL"
    ))
    t.append(run_check(
        "orders_valid_status", "orders",
        # Full set of statuses present in Olist dataset
        """SELECT COUNT(*) AS result FROM sv_orders
           WHERE order_status NOT IN
           ('delivered','shipped','processing','canceled',
            'invoiced','approved','unavailable','created')"""
    ))
    t.append(run_check(
        "orders_no_duplicate_order_id", "orders",
        # merge_keys = [order_id] in table_config
        """SELECT COUNT(*) AS result FROM (
               SELECT order_id, COUNT(*) AS cnt
               FROM sv_orders GROUP BY order_id HAVING cnt > 1
           )"""
    ))
    t.append(run_check(
        "orders_approved_not_before_purchase", "orders",
        """SELECT COUNT(*) AS result FROM sv_orders
           WHERE order_approved_at IS NOT NULL
             AND order_purchase_timestamp IS NOT NULL
             AND order_approved_at < order_purchase_timestamp"""
    ))
    t.append(run_check(
        "orders_delivered_not_before_purchase", "orders",
        """SELECT COUNT(*) AS result FROM sv_orders
           WHERE order_delivered_customer_date IS NOT NULL
             AND order_purchase_timestamp IS NOT NULL
             AND order_delivered_customer_date < order_purchase_timestamp"""
    ))

    all_results.extend(t)

    orders_bad = (
        col("order_id").isNull()
        | col("customer_id").isNull()
        | col("order_purchase_timestamp").isNull()
        | ~col("order_status").isin(
            "delivered", "shipped", "processing", "canceled",
            "invoiced", "approved", "unavailable", "created"
        )
        | (
            col("order_approved_at").isNotNull()
            & col("order_purchase_timestamp").isNotNull()
            & (col("order_approved_at") < col("order_purchase_timestamp"))
        )
        | (
            col("order_delivered_customer_date").isNotNull()
            & col("order_purchase_timestamp").isNotNull()
            & (col("order_delivered_customer_date") < col("order_purchase_timestamp"))
        )
    )
    quarantine_and_clean("orders", df_orders, orders_bad, t)


# ══════════════════════════════════════════════════════════════════════════════
# TABLE 2: silver.order_items
#
# Source: order_items_dataset.csv
# table_config cleaning_rules applied in Silver:
#   price         → cast_double
#   freight_value → cast_double
# Merge keys: [order_id, order_item_id]
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 60)
print("TABLE: silver.order_items")
print("─" * 60)

try:
    df_items = spark.read.format("delta").load(f"{S3_DELTA_SILVER}/order_items")
    df_items.createOrReplaceTempView("sv_order_items")
    items_ok = True
except Exception as e:
    print(f"  [SKIP] Cannot read silver.order_items: {e}")
    items_ok = False

if items_ok:
    t = []

    t.append(run_check(
        "order_items_no_null_order_id", "order_items",
        "SELECT COUNT(*) AS result FROM sv_order_items WHERE order_id IS NULL"
    ))
    t.append(run_check(
        "order_items_no_null_product_id", "order_items",
        "SELECT COUNT(*) AS result FROM sv_order_items WHERE product_id IS NULL"
    ))
    t.append(run_check(
        "order_items_no_null_seller_id", "order_items",
        "SELECT COUNT(*) AS result FROM sv_order_items WHERE seller_id IS NULL"
    ))
    t.append(run_check(
        "order_items_price_positive", "order_items",
        # price is cast_double — NULL = cast failed in Silver
        "SELECT COUNT(*) AS result FROM sv_order_items WHERE price IS NULL OR price <= 0"
    ))
    t.append(run_check(
        "order_items_freight_non_negative", "order_items",
        # freight_value = 0 is valid (free shipping) — only negative is bad
        "SELECT COUNT(*) AS result FROM sv_order_items WHERE freight_value IS NULL OR freight_value < 0"
    ))
    t.append(run_check(
        "order_items_price_not_extreme", "order_items",
        # Max real Olist price ~6,735 BRL — flag > 10,000 as data error
        "SELECT COUNT(*) AS result FROM sv_order_items WHERE price > 10000"
    ))
    t.append(run_check(
        "order_items_item_id_positive", "order_items",
        # order_item_id is line number within order — starts at 1
        "SELECT COUNT(*) AS result FROM sv_order_items WHERE order_item_id IS NULL OR order_item_id < 1"
    ))
    t.append(run_check(
        "order_items_no_duplicate_line", "order_items",
        # merge_keys = [order_id, order_item_id] in table_config
        """SELECT COUNT(*) AS result FROM (
               SELECT order_id, order_item_id, COUNT(*) AS cnt
               FROM sv_order_items GROUP BY 1, 2 HAVING cnt > 1
           )"""
    ))

    all_results.extend(t)

    items_bad = (
        col("order_id").isNull()
        | col("product_id").isNull()
        | col("seller_id").isNull()
        | col("price").isNull()         | (col("price") <= 0)
        | col("freight_value").isNull() | (col("freight_value") < 0)
        | (col("price") > 10000)
        | col("order_item_id").isNull() | (col("order_item_id") < 1)
    )
    quarantine_and_clean("order_items", df_items, items_bad, t)


# ══════════════════════════════════════════════════════════════════════════════
# TABLE 3: silver.order_payments
#
# Source: order_payments_dataset.csv
# table_config cleaning_rules applied in Silver:
#   payment_type         → replace_value ('not_defined' → 'unknown')
#   payment_value        → cast_double
#   payment_installments → cast_int
# Merge keys: [order_id, payment_sequential]
# Dedup keys: [] (table_config dedup_keys is empty for this table)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 60)
print("TABLE: silver.order_payments")
print("─" * 60)

try:
    df_payments = spark.read.format("delta").load(f"{S3_DELTA_SILVER}/order_payments")
    df_payments.createOrReplaceTempView("sv_order_payments")
    payments_ok = True
except Exception as e:
    print(f"  [SKIP] Cannot read silver.order_payments: {e}")
    payments_ok = False

if payments_ok:
    t = []

    t.append(run_check(
        "order_payments_no_null_order_id", "order_payments",
        "SELECT COUNT(*) AS result FROM sv_order_payments WHERE order_id IS NULL"
    ))
    t.append(run_check(
        "order_payments_valid_payment_type", "order_payments",
        # Silver replace_value turns 'not_defined' → 'unknown', so 'unknown' is valid
        """SELECT COUNT(*) AS result FROM sv_order_payments
           WHERE payment_type NOT IN
           ('credit_card','boleto','voucher','debit_card','unknown')"""
    ))
    t.append(run_check(
        "order_payments_value_non_negative", "order_payments",
        # payment_value = 0 is valid (voucher covers full amount)
        "SELECT COUNT(*) AS result FROM sv_order_payments WHERE payment_value IS NULL OR payment_value < 0"
    ))
    t.append(run_check(
        "order_payments_installments_at_least_one", "order_payments",
        # cast_int → NULL means original value was non-numeric
        "SELECT COUNT(*) AS result FROM sv_order_payments WHERE payment_installments IS NULL OR payment_installments < 1"
    ))
    t.append(run_check(
        "order_payments_sequential_at_least_one", "order_payments",
        "SELECT COUNT(*) AS result FROM sv_order_payments WHERE payment_sequential IS NULL OR payment_sequential < 1"
    ))

    all_results.extend(t)

    payments_bad = (
        col("order_id").isNull()
        | ~col("payment_type").isin(
            "credit_card", "boleto", "voucher", "debit_card", "unknown"
        )
        | col("payment_value").isNull()        | (col("payment_value") < 0)
        | col("payment_installments").isNull() | (col("payment_installments") < 1)
        | col("payment_sequential").isNull()   | (col("payment_sequential") < 1)
    )
    quarantine_and_clean("order_payments", df_payments, payments_bad, t)


# ══════════════════════════════════════════════════════════════════════════════
# TABLE 4: silver.order_reviews
#
# Source: order_reviews_dataset.csv
# table_config cleaning_rules applied in Silver:
#   review_score           → cast_int   (valid range: 1–5)
#   review_comment_title   → fill_null  (default: "")
#   review_comment_message → fill_null  (default: "")
#
# NOT in cleaning_rules (so not cast by Silver):
#   review_creation_date, review_answer_timestamp
#   These columns stay as whatever inferSchema decided (string or timestamp).
#   We do NOT check temporal ordering on them to avoid type-mismatch errors.
#
# Merge key: review_id
# Dedup key: review_id
#
# NOTE on review_score DQ:
#   gold_engine uses coalesce(review_score, 0.0) on the aggregated avg
#   so NULLs don't block Gold. But score outside 1–5 is still flagged here.
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 60)
print("TABLE: silver.order_reviews")
print("─" * 60)

try:
    df_reviews = spark.read.format("delta").load(f"{S3_DELTA_SILVER}/order_reviews")
    df_reviews.createOrReplaceTempView("sv_order_reviews")
    reviews_ok = True
except Exception as e:
    print(f"  [SKIP] Cannot read silver.order_reviews: {e}")
    reviews_ok = False

if reviews_ok:
    t = []

    t.append(run_check(
        "order_reviews_no_null_review_id", "order_reviews",
        "SELECT COUNT(*) AS result FROM sv_order_reviews WHERE review_id IS NULL"
    ))
    t.append(run_check(
        "order_reviews_no_null_order_id", "order_reviews",
        "SELECT COUNT(*) AS result FROM sv_order_reviews WHERE order_id IS NULL"
    ))
    t.append(run_check(
        "order_reviews_valid_score", "order_reviews",
        # cast_int in Silver — NULL means original was non-numeric
        # Valid Olist scores: 1, 2, 3, 4, 5
        "SELECT COUNT(*) AS result FROM sv_order_reviews WHERE review_score NOT BETWEEN 1 AND 5"
    ))
    t.append(run_check(
        "order_reviews_no_duplicate_review_id", "order_reviews",
        # merge_keys = dedup_keys = [review_id] in table_config
        """SELECT COUNT(*) AS result FROM (
               SELECT review_id, COUNT(*) AS cnt
               FROM sv_order_reviews GROUP BY review_id HAVING cnt > 1
           )"""
    ))

    all_results.extend(t)

    # Bad condition: null IDs + review_score outside 1–5
    # We do NOT check review_creation_date / review_answer_timestamp ordering
    # because those columns are not in cleaning_rules and their inferred type
    # varies (string vs timestamp) — checking would cause type errors.
    reviews_bad = (
        col("review_id").isNull()
        | col("order_id").isNull()
        | ~col("review_score").between(1, 5)
    )
    quarantine_and_clean("order_reviews", df_reviews, reviews_bad, t)


# ══════════════════════════════════════════════════════════════════════════════
# CROSS-TABLE: Referential integrity checks (informational — no quarantine)
#
# Gold engine uses LEFT JOINs so orphan rows don't block Gold.
# These checks are logged for visibility only.
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 60)
print("CROSS-TABLE: Referential integrity (log only, no quarantine)")
print("─" * 60)

if items_ok and orders_ok:
    all_results.append(run_check(
        "ref_items_orphan_in_orders", "cross_table",
        """SELECT COUNT(*) AS result
           FROM sv_order_items i
           LEFT JOIN sv_orders o ON i.order_id = o.order_id
           WHERE o.order_id IS NULL"""
    ))

if payments_ok and orders_ok:
    all_results.append(run_check(
        "ref_payments_orphan_in_orders", "cross_table",
        """SELECT COUNT(*) AS result
           FROM sv_order_payments p
           LEFT JOIN sv_orders o ON p.order_id = o.order_id
           WHERE o.order_id IS NULL"""
    ))

if reviews_ok and orders_ok:
    all_results.append(run_check(
        "ref_reviews_orphan_in_orders", "cross_table",
        """SELECT COUNT(*) AS result
           FROM sv_order_reviews r
           LEFT JOIN sv_orders o ON r.order_id = o.order_id
           WHERE o.order_id IS NULL"""
    ))


# ══════════════════════════════════════════════════════════════════════════════
# SAVE DQ LOG → silver._dq_log (Delta, append)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 60)
print("Saving DQ results → silver._dq_log")

df_log = (
    spark.createDataFrame(all_results)
    .withColumn("run_timestamp", current_timestamp())
    .withColumn("batch_number",  lit(batch_number))
)

(
    df_log.write
    .format("delta")
    .mode("append")
    .option("path", DQ_LOG_PATH)
    .option("mergeSchema", "true")
    .saveAsTable("silver._dq_log")
)
print(f"Saved {len(all_results)} records → silver._dq_log")


# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
pass_count  = sum(1 for r in all_results if r["status"] == "PASS")
fail_count  = sum(1 for r in all_results if r["status"] == "FAIL")
error_count = sum(1 for r in all_results if r["status"] == "ERROR")

print("")
print("=" * 60)
print(f"DQ SUMMARY — batch {batch_number}")
print("=" * 60)
print(f"Total : {len(all_results)}  |  Passed: {pass_count}  |  Failed: {fail_count}  |  Errors: {error_count}")

if fail_count > 0 or error_count > 0:
    print("\nFAILED / ERROR TESTS:")
    for r in all_results:
        if r["status"] in ("FAIL", "ERROR"):
            print(f"  {r['status']:5s} [{r['table_name']:20s}] {r['test_name']}: {r['result_value']} bad rows")

if fail_count == 0 and error_count == 0:
    print("\n✅ ALL DQ CHECKS PASSED — Silver is clean. Proceeding to Gold.")
else:
    print(f"\n⚠️  {fail_count} check(s) failed. Bad rows quarantined.")
    print("    Silver tables now contain clean rows only.")
    print("    Gold will receive clean data.")
    print("    Inspect silver._dq_quarantine for bad rows.")


# ══════════════════════════════════════════════════════════════════════════════
# STOP if any table was entirely quarantined
# ══════════════════════════════════════════════════════════════════════════════
if stop_pipeline:
    raise Exception(
        f"DQ FATAL: One or more Silver tables for batch {batch_number} "
        f"had ALL rows quarantined — nothing clean to promote to Gold. "
        f"Pipeline stopped. Check silver._dq_quarantine for details."
    )